# Power System Fault Dataset — Exploratory Data Analysis

This notebook provides an interactive exploration of the synthetic fault dataset generated by `src/data/generator.py`.

**Run `python run_pipeline.py --generate-only` first to create the dataset.**

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import get_config
from src.data.generator import FAULT_CLASSES, CHANNEL_NAMES

cfg = get_config('../config/config.yaml')
synthetic_dir = Path('../data/synthetic')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
print('Setup complete')

In [ ]:
# Load dataset
df = pd.read_csv(synthetic_dir / 'raw_timeseries.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 4))
counts = df.groupby(['label', 'fault_name']).size().reset_index(name='count')
ax.bar([f"{r.label}: {r.fault_name}" for _, r in counts.iterrows()], counts['count'])
ax.set_xlabel('Fault Class')
ax.set_ylabel('Sample Count')
ax.set_title('Class Distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(counts.to_string(index=False))

In [ ]:
# Voltage waveforms per fault class (1000-sample excerpt)
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for idx, (cls_id, cls_name) in enumerate(FAULT_CLASSES.items()):
    subset = df[df['label'] == cls_id].iloc[:1000]
    ax = axes[idx]
    for ch in ['Va', 'Vb', 'Vc']:
        ax.plot(subset[ch].values, alpha=0.75, linewidth=0.8, label=ch)
    ax.set_title(f'Class {cls_id}: {cls_name}', fontsize=10)
    ax.set_ylabel('Voltage (V)')
    ax.legend(fontsize=8)
    ax.set_ylim(-400, 400)

if len(FAULT_CLASSES) < len(axes):
    axes[-1].set_visible(False)

plt.suptitle('Phase Voltage Waveforms by Fault Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Channel statistics by fault class
stats_df = df.groupby('label')[CHANNEL_NAMES].agg(['mean', 'std'])
stats_df.columns = ['_'.join(c) for c in stats_df.columns]
print('Mean voltage Va by class:')
print(stats_df[['Va_mean', 'Va_std']].to_string())

In [ ]:
# THD distribution by fault class
fig, ax = plt.subplots(figsize=(10, 5))
for cls_id, cls_name in FAULT_CLASSES.items():
    subset = df[df['label'] == cls_id]['THD_V']
    ax.hist(subset, bins=50, alpha=0.5, label=f'{cls_id}:{cls_name}', density=True)
ax.set_xlabel('THD_V (%)')
ax.set_ylabel('Density')
ax.set_title('THD_V Distribution by Fault Class')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of sensor channels
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[CHANNEL_NAMES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Sensor Channel Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Neutral current by fault class (key SLG discriminator)
fig, ax = plt.subplots(figsize=(10, 4))
df.boxplot(column='In', by='fault_name', ax=ax)
ax.set_title('Neutral Current (In) Distribution by Fault Class')
ax.set_xlabel('Fault Class')
ax.set_ylabel('In (A)')
plt.suptitle('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()